# threadlepy tutorial

This notebook introduces `threadlepy` with a deliberately step-by-step layout. Most single API calls are placed in their own code cell so you can run them one at a time, inspect the output, and isolate errors easily.

The final section shows how to compose several functions into a small workflow. It mirrors the `th_attr_walk()` example in `threadleR/scripts/01_get_started.R`: a Python attribute-filtered random walk that chooses candidate nodes by node attributes.

You need a working Threadle CLI executable. This tutorial assumes `threadle` is available on `PATH`. If it is not, pass the full executable path to `th.start_threadle("/full/path/to/threadle")`.


## 1. Imports and startup

Import the package, configure client output, and start a background Threadle JSON process. In notebooks, explicit `start` / `stop` cells are convenient. In scripts, `with threadlepy.session():` is usually cleaner.


In [ ]:
import os


In [ ]:
from pathlib import Path


In [ ]:
import threadlepy


In [ ]:
import threadlepy.commands as th


In [ ]:
threadlepy.__version__


In [ ]:
threadlepy.is_available()


In [ ]:
th.is_available()


In [ ]:
threadlepy.configure(print_messages=False, timeout=1800)


In [ ]:
th.start_threadle()


In [ ]:
tutorial_workdir = th.get_workdir()


In [ ]:
tutorial_workdir


## 2. Working directory and example data

These functions inspect or set the working directory seen by the Threadle backend, and copy bundled example files into a local project folder.


In [ ]:
th.get_workdir()


In [ ]:
example_dir = th.stage_examples_to_wd("threadle_examples", overwrite=True)


In [ ]:
example_dir


In [ ]:
th.set_workdir(example_dir)


In [ ]:
th.dir()


In [ ]:
th.sync_wd()


In [ ]:
th.get_workdir()


`load_examples()` is the easiest way to load packaged data. Here we load `mynet` and `lazega`. The returned values are lightweight handles; the actual data live inside the Threadle backend session.


In [ ]:
examples = th.load_examples(["mynet", "lazega"], assign=False)


In [ ]:
mynet_nodeset = examples["mynet_nodeset"]


In [ ]:
mynet = examples["mynet"]


In [ ]:
lazega_nodeset = examples["lazega_nodeset"]


In [ ]:
lazega = examples["lazega"]


You can also load a nodeset or network directly from a file.


In [ ]:
loaded_nodes = th.load_file("loaded_nodes", os.path.join(example_dir, "mynet_nodesetfile.tsv"), type="nodeset")


In [ ]:
loaded_net = th.load_file("loaded_net", os.path.join(example_dir, "mynet.tsv"), type="network")


## 3. Session metadata and low-level commands

`inventory()` and `i()` are equivalent. `cmd()` is a low-level escape hatch for calling Threadle commands that do not yet have a typed Python wrapper.


In [ ]:
th.inventory()


In [ ]:
th.i()


In [ ]:
th.info(mynet)


In [ ]:
th.preview(mynet)


In [ ]:
th.cmd("info", args={"structure": mynet})


## 4. Create scratch structures

Create a small scratch nodeset and network for mutating examples. This keeps the bundled example data easy to inspect.


In [ ]:
scratch_nodes = th.create_nodeset("tutorial_nodes", name="Tutorial nodes", createnodes=5)


In [ ]:
scratch_net = th.create_network("tutorial_net", scratch_nodes, name="Tutorial network")


In [ ]:
th.info(scratch_nodes)


In [ ]:
th.info(scratch_net)


## 5. Node functions

Node functions work on either a nodeset or a network, because a Threadle network carries an underlying nodeset.


In [ ]:
th.get_nbr_nodes(scratch_nodes)


In [ ]:
th.get_all_nodes(scratch_nodes, offset=0, limit=10)


In [ ]:
th.get_nodeid_by_index(scratch_nodes, index=0)


In [ ]:
th.get_random_node(scratch_nodes)


In [ ]:
th.add_node(scratch_nodes, nodeid=10)


In [ ]:
th.get_all_nodes(scratch_nodes, offset=0, limit=20)


In [ ]:
th.remove_node(scratch_nodes, nodeid=10)


In [ ]:
th.get_all_nodes(scratch_nodes, offset=0, limit=20)


## 6. Node attribute functions

Define an attribute, set values, read values, summarize values, and delete values. `generate_attr()` is useful for quickly creating random attribute values.


In [ ]:
th.define_attr(scratch_nodes, "score", "int")


In [ ]:
th.set_attr(scratch_nodes, nodeid=1, attrname="score", attrvalue=7)


In [ ]:
th.set_attr(scratch_nodes, nodeid=2, attrname="score", attrvalue=9)


In [ ]:
th.get_attr(scratch_nodes, nodeid=1, attrname="score")


In [ ]:
th.get_attrs(scratch_nodes, nodes=[1, 2, 3], attrname="score")


In [ ]:
th.get_attr_summary(scratch_nodes, attrname="score")


In [ ]:
th.remove_attr(scratch_nodes, nodeid=1, attrname="score")


In [ ]:
th.undefine_attr(scratch_nodes, "score")


In [ ]:
th.generate_attr(scratch_nodes, "random_score", attrtype="int", min=1, max=10)


In [ ]:
th.get_attr_summary(scratch_nodes, attrname="random_score")


The example data already include attributes, so you can query them directly.


In [ ]:
th.get_attr(mynet_nodeset, nodeid=123, attrname="gender")


In [ ]:
th.get_attrs(mynet_nodeset, nodes=[123, 234, 345], attrname="gender")


In [ ]:
th.get_attr_summary(mynet_nodeset, attrname="gender")


## 7. One-mode layers and edge functions

Add a one-mode layer to the scratch network, then add, inspect, remove, and clear edges.


In [ ]:
th.add_layer(scratch_net, "friendship", mode=1, directed=False, valuetype="binary", selfties=False)


In [ ]:
th.add_edge(scratch_net, "friendship", node1id=1, node2id=2)


In [ ]:
th.add_edge(scratch_net, "friendship", node1id=2, node2id=3)


In [ ]:
th.check_edge(scratch_net, "friendship", node1id=1, node2id=2)


In [ ]:
th.get_edge(scratch_net, "friendship", node1id=1, node2id=2)


In [ ]:
th.get_all_edges(scratch_net, "friendship", offset=0, limit=10)


In [ ]:
th.get_random_edge(scratch_net, "friendship")


In [ ]:
th.remove_edge(scratch_net, "friendship", node1id=2, node2id=3)


In [ ]:
th.clear_layer(scratch_net, "friendship")


In [ ]:
th.get_all_edges(scratch_net, "friendship", offset=0, limit=10)


## 8. Two-mode layers, hyperedges, and affiliations

A two-mode layer represents events, organizations, or categories as hyperedges. Affiliations connect nodes to those hyperedges.


In [ ]:
th.add_layer(scratch_net, "events", mode=2)


In [ ]:
th.add_hyper(scratch_net, "events", hypername="seminar", nodes=[1, 2])


In [ ]:
th.add_aff(scratch_net, "events", nodeid=3, hypername="seminar")


In [ ]:
th.get_all_hyperedges(scratch_net, "events", offset=0, limit=10)


In [ ]:
th.get_hyperedge_nodes(scratch_net, "events", hypername="seminar")


In [ ]:
th.get_node_hyperedges(scratch_net, "events", nodeid=3)


In [ ]:
th.remove_aff(scratch_net, "events", nodeid=3, hypername="seminar")


In [ ]:
th.remove_hyper(scratch_net, "events", hypername="seminar")


The example `mynet` network also includes a two-mode `work` layer that you can query directly.


In [ ]:
th.get_all_hyperedges(mynet, "work", offset=0, limit=10)


In [ ]:
th.get_hyperedge_nodes(mynet, "work", hypername="ias")


In [ ]:
th.get_node_hyperedges(mynet, "work", nodeid=123)


## 9. Alters, paths, and network measures

These functions usually operate on an existing one-mode layer. `mynet` includes the one-mode layers `kinship` and `trade`.


In [ ]:
th.get_edge(mynet, "trade", node1id=123, node2id=345)


In [ ]:
th.check_edge(mynet, "kinship", node1id=123, node2id=345)


In [ ]:
th.get_all_edges(mynet, "trade", offset=0, limit=10)


In [ ]:
th.get_random_edge(mynet, "trade")


In [ ]:
th.get_node_alters(mynet, nodeid=345, layernames="trade", direction="both")


In [ ]:
th.get_degree(mynet, nodeid=345, layernames="trade", direction="both")


In [ ]:
th.get_random_alter(mynet, nodeid=345, layernames="trade", direction="both")


In [ ]:
th.shortest_path(mynet, node1id=123, node2id=567, layernames="kinship")


In [ ]:
paths_by_gender = th.shortest_paths("paths_by_gender", mynet, attrname="gender", layernames="kinship")


In [ ]:
th.info(paths_by_gender)


In [ ]:
th.density(mynet, "kinship")


In [ ]:
th.components(mynet, "kinship", attrname="kinship_component")


In [ ]:
th.degree(mynet, "kinship", attrname="kinship_degree", direction="both")


In [ ]:
th.get_attr_summary(mynet, attrname="kinship_degree")


## 10. Filtering, subnetworks, and layer transforms

These functions create new nodesets, networks, or layers from existing data.


In [ ]:
female_nodeset = th.filter("female_nodeset", mynet_nodeset, attrname="gender", cond="eq", attrvalue="f")


In [ ]:
female_nodeset_2 = th.filter_nodeset("female_nodeset_2", mynet_nodeset, attrname="gender", cond="eq", attrvalue="f")


In [ ]:
female_mynet = th.subnet("female_mynet", mynet, female_nodeset)


In [ ]:
th.info(female_mynet)


In [ ]:
th.symmetrize(mynet, "trade", method="max", newlayername="trade_sym")


In [ ]:
th.dichotomize(mynet, "trade", cond="ge", threshold=1000, truevalue=1, falsevalue=0, newlayername="trade_hi")


In [ ]:
th.project_two_mode(mynet, "work", method="count", newlayername="work_projected")


In [ ]:
th.get_edge(mynet, "trade_sym", node1id=123, node2id=345)


In [ ]:
th.get_edge(mynet, "trade_hi", node1id=890, node2id=234)


In [ ]:
th.get_edge(mynet, "work_projected", node1id=123, node2id=456)


In [ ]:
th.pack(mynet, layername="kinship")


In [ ]:
th.unpack(mynet, layername="kinship")


## 11. Random generation, settings, and seeds

Use these functions to configure backend behavior and generate random ties in a layer.


In [ ]:
th.setting("verbose", False)


In [ ]:
th.random_seed(42)


In [ ]:
random_nodes = th.create_nodeset("random_nodes", createnodes=20)


In [ ]:
random_net = th.create_network("random_net", random_nodes)


In [ ]:
th.add_layer(random_net, "er_layer", mode=1, directed=False, valuetype="binary", selfties=False)


In [ ]:
th.generate(random_net, "er_layer", type="er", p=0.15)


In [ ]:
th.get_all_edges(random_net, "er_layer", offset=0, limit=10)


## 12. Import, export, save, and scripts

This section writes a few temporary files, so it includes a small amount of normal Python file handling. Threadle API calls are still kept in mostly separate cells.


In [ ]:
output_dir = Path(tutorial_workdir) / "threadle_tutorial_outputs"


In [ ]:
output_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
kinship_edges_file = output_dir / "mynet_kinship_edges.tsv"


In [ ]:
mynet_gexf_file = output_dir / "mynet_kinship.gexf"


In [ ]:
scratch_nodes_file = output_dir / "scratch_nodes_saved.tsv"


In [ ]:
import_edges_file = output_dir / "import_edges.tsv"


In [ ]:
script_file = output_dir / "threadle_script.txt"


In [ ]:
th.export_layer(mynet, "kinship", file=str(kinship_edges_file), header=True, sep="\t")


In [ ]:
kinship_edges_file.read_text(encoding="utf-8").splitlines()[:3]


In [ ]:
th.export(mynet, file=str(mynet_gexf_file), layername="kinship", format="gexf")


In [ ]:
th.save_file(scratch_nodes, file=str(scratch_nodes_file))


In [ ]:
scratch_nodes_copy = th.load_file("scratch_nodes_copy", file=str(scratch_nodes_file), type="nodeset")


In [ ]:
th.get_nbr_nodes(scratch_nodes_copy)


In [ ]:
import_edges_file.write_text("1\t2\t4\n2\t3\t5\n", encoding="utf-8")


In [ ]:
th.add_layer(scratch_net, "imported_edges", mode=1, directed=False, valuetype="valued", selfties=False)


In [ ]:
th.import_layer(scratch_net, "imported_edges", file=str(import_edges_file), format="edgelist", node1col=0, node2col=1, valuecol=2, header=False, sep="\t", addmissingnodes=True)


In [ ]:
th.get_edge(scratch_net, "imported_edges", node1id=1, node2id=2)


In [ ]:
script_file.write_text(
    "script_nodes = createnodeset(name=script_nodes, createnodes=3)\n"
    "script_net = createnetwork(name=script_net, nodeset=script_nodes)\n"
    "addlayer(network=script_net, layername=script_layer, mode=1, directed=false, valuetype=binary, selfties=false)\n",
    encoding="utf-8",
)


In [ ]:
th.load_script(str(script_file))


In [ ]:
th.info("script_net")


## 13. Delete objects and layers

Deletion examples are best placed near the end of a tutorial. Here we remove a temporary layer and object, then later show `delete_all()`.


In [ ]:
th.add_layer(scratch_net, "temporary", mode=1)


In [ ]:
th.remove_layer(scratch_net, "temporary")


In [ ]:
delete_me = th.create_nodeset("delete_me", createnodes=1)


In [ ]:
th.delete(delete_me)


## 14. Complete example: attribute-filtered random walk

This section follows the structure of `th_attr_walk()` in `threadleR/scripts/01_get_started.R`:

1. Choose a starting node; if none is supplied, use `get_random_node()`.
2. At each step, use `get_node_alters()` to find the current node's neighbors.
3. Use `get_attr()` to read each neighbor's attribute, then prefer neighbors whose attribute is in `keep`.
4. If no neighbor matches the attribute filter, use `fallback` to decide whether to stop or sample from all neighbors.
5. Return one row per step with `step`, `nodeid`, and the selected attribute value.


In [ ]:
import random


In [ ]:
def attr_walk(
    network,
    start=None,
    steps=30,
    layernames=None,
    direction="both",
    attrname="gender",
    keep=("f",),
    unique=True,
    fallback="any",
):
    if fallback not in {"any", "stop"}:
        raise ValueError('fallback must be "any" or "stop"')

    cur = th.get_random_node(network) if start is None else start
    path = [cur]

    for _ in range(steps):
        alters = th.get_node_alters(
            network=network,
            nodeid=cur,
            layernames=layernames,
            direction=direction,
            unique=unique,
        )

        if not alters:
            if fallback == "stop":
                break
            path.append(None)
            break

        attr_values = {
            nodeid: str(th.get_attr(network, nodeid=nodeid, attrname=attrname))
            for nodeid in alters
        }
        candidates = [nodeid for nodeid in alters if attr_values[nodeid] in keep]

        if candidates:
            nxt = random.choice(candidates)
        elif fallback == "stop":
            break
        else:
            nxt = random.choice(alters)

        cur = nxt
        path.append(cur)

    return [
        {
            "step": step,
            "nodeid": nodeid,
            "attr": None if nodeid is None else th.get_attr(network, nodeid=nodeid, attrname=attrname),
        }
        for step, nodeid in enumerate(path)
    ]


First confirm the relevant layer and attribute names. Earlier sections mostly use `mynet` with `gender`; the R script example uses `lazega` with `Gender` and `friends`.


In [ ]:
th.info(lazega)


In [ ]:
th.preview(lazega)


In [ ]:
walk1 = attr_walk(network=lazega, steps=20, layernames="friends", direction="both", attrname="Gender", keep=("M",), fallback="any")


In [ ]:
walk1


If you want tabular display, convert the result to a pandas DataFrame.


In [ ]:
try:
    import pandas as pd
    pd.DataFrame(walk1)
except ImportError:
    walk1


The cells below keep examples for Threadle's built-in random-walk summary commands. They are different from `attr_walk()`: `attr_walk()` returns one concrete path, while `rwdistances()` and `rwfpt()` return category-level result networks or statistics.


In [ ]:
rw_distances = th.rwdistances("rw_distances", lazega, attrname="Gender", maxsteps=30, layernames="friends", walkfactor=0.5, balanced=False, weighted=False, backtrack=False, savesteps=False)


In [ ]:
th.info(rw_distances)


In [ ]:
rw_fpt = th.rwfpt("rw_fpt", lazega, attrname="Gender", maxsteps=30, layernames="friends", walkfactor=0.5, minpairobs=1, balanced=False, weighted=False)


In [ ]:
th.info(rw_fpt)


In [ ]:
rw_fpt_hist = th.rwfpt("rw_fpt_hist", lazega, attrname="Gender", maxsteps=30, layernames="friends", walkfactor=0.5, minpairobs=1, balanced=False, weighted=False, return_histograms=True)


In [ ]:
rw_fpt_hist


## 15. End the session

If you only want to clear backend objects, run `delete_all()`. When you are done with the notebook, stop the background Threadle process.


In [ ]:
th.delete_all()


In [ ]:
th.i()


In [ ]:
th.stop_threadle()
